### Imports

In [1]:
import sys
import os
from pathlib import Path

# Add parent directory to sys.path so hvac module is discoverable
# (notebook runs from blog_posts/, need to go up one level to anomaly_detection/)
sys.path.insert(0, os.path.dirname(os.getcwd()))

from hvac.utils import hvac_data_gen as hvdg
from datetime import datetime, timedelta
import plotly.express as px
import stumpy
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from hvac.utils import visual as vis
from hvac.utils import euclidean_dist as eucl_dist 

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

# Data

In [ ]:
hvac_dataset = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))

for anom in ['lag', 'frequency', 'amplitude']:
    fig1 = vis.plot_container_anomaly_timeseries(hvac_dataset, anomaly_type=anom, num_containers=1)
    fig1.show()

fig2 = vis.plot_anomaly_type_distribution(hvac_dataset)
fig2.show()

[TODO] Add high level conclusions about the distribution and types of anomalies. A brief note on which methods are expected to work. 

## Day Level flags
- Business Objective: Flag container_day
- Assume only one unit failing 

In [ ]:
hvac_dataset["day"] = hvac_dataset["timestamp_et"].dt.date                                                                                                                                                    
day_labels = (
      hvac_dataset.groupby(["container_id", "day"]).apply(
        lambda grp: pd.Series({
                'anomaly': grp['anomaly'].max(),
                'type': 'normal' if (grp['anomaly_type'] == 'normal').all() else (grp.loc[grp['anomaly_type'] != 'normal', 'anomaly_type']).unique()[0]
            })
      )
      
    #   .reset_index(name="label")
  )

In [ ]:
day_labels[day_labels['type'] != 'normal'].head()

# Anomaly Detection

## Correlation
- feature calculation: 
  - rolling window (1-correlation) averaged over the day.
  - take max of the three pairs
- anomaly_score = correlation

In [ ]:
corr_df = eucl_dist.compute_pairwise_correlations(hvac_dataset)
corr_scores_df = eucl_dist.score_anomalies(corr_df, strategy="mad", model_name="correlation_mad")

In [ ]:
corr_scores_labels_df = corr_scores_df.merge(day_labels.reset_index(), on=["container_id", "day"], how="left")


In [ ]:
corr_scores_labels_df.head()

In [ ]:
px.histogram(corr_scores_labels_df, x='anomaly_score', color='type')


## Euclidean Distance
- feature to use: rolling euclidean distance averaged over the day
- anomaly_score pair_day_level = (d_(ij) - median)/(MAD* 1.4826)
- take max of the score of three pairs as container-day level  anomaly score

In [10]:

dist_df = eucl_dist.compute_pairwise_distances(hvac_dataset)
eucl_scores_df = eucl_dist.score_anomalies(dist_df, "iqr")

In [11]:

eucl_scores_labels_df = eucl_scores_df.merge(day_labels.reset_index(), on=["container_id", "day"], how="left")

In [12]:
eucl_scores_labels_df.head()

,container_id,day,anomaly_score,model,anomaly,type
0,0,2026-01-01,-0.722585,euclidean_distance_iqr,False,normal
1,0,2026-01-02,-0.722915,euclidean_distance_iqr,False,normal
2,0,2026-01-03,-0.722953,euclidean_distance_iqr,False,normal
3,0,2026-01-04,-0.722740,euclidean_distance_iqr,False,normal
4,0,2026-01-05,-0.722561,euclidean_distance_iqr,False,normal


## AUC-PR Compare

In [13]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = {"euclidean_distance_iqr": "#5470C6", "correlation_mad": "#EE6666"}

# PR curves
fig_pr = make_subplots(rows=1, cols=3, subplot_titles=["Lag", "Frequency", "Amplitude"],
                       shared_yaxes=True)

for col, atype in enumerate(["lag", "frequency", "amplitude"], 1):
    for df in [eucl_scores_labels_df, corr_scores_labels_df]:
        mask = df["type"].isin([atype, "normal"])
        subset = df[mask]
        lab = (subset["type"] == atype).astype(int)

        if lab.sum() == 0:
            continue

        prec, rec, _ = precision_recall_curve(lab, subset["anomaly_score"])
        ap = average_precision_score(lab, subset["anomaly_score"])
        model = df['model'].iloc[0]

        fig_pr.add_trace(go.Scatter(x=rec, y=prec, mode="lines",
                    name=model, line=dict(color=colors[model]),
                    legendgroup=model, showlegend=(col == 1)),
            row=1, col=col
        )

    # Baseline
    baseline = (eucl_scores_labels_df[eucl_scores_labels_df["type"].isin([atype, "normal"])]["type"] == atype).mean()
    fig_pr.add_hline(y=baseline, line_dash="dot", line_color="gray", row=1, col=col)

fig_pr.update_xaxes(title_text="Recall", range=[0, 1])
fig_pr.update_yaxes(title_text="Precision", range=[0, 1], col=1)
fig_pr.update_layout(height=400, width=1200, title="AUC-PR Curves by Anomaly Type")
fig_pr.show()

# ROC curves
fig_roc = make_subplots(rows=1, cols=3, subplot_titles=["Lag", "Frequency", "Amplitude"],
                        shared_yaxes=True)

for col, atype in enumerate(["lag", "frequency", "amplitude"], 1):
    for df in [eucl_scores_labels_df, corr_scores_labels_df]:
        mask = df["type"].isin([atype, "normal"])
        subset = df[mask]
        lab = (subset["type"] == atype).astype(int)

        if lab.sum() == 0:
            continue

        fpr, tpr, _ = roc_curve(lab, subset["anomaly_score"])
        auc = roc_auc_score(lab, subset["anomaly_score"])
        model = df['model'].iloc[0]

        fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                    name=model, line=dict(color=colors[model]),
                    legendgroup=model, showlegend=(col == 1)),
            row=1, col=col
        )

    # Diagonal
    fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                line=dict(color="gray", dash="dot"), showlegend=False),
        row=1, col=col
    )

fig_roc.update_xaxes(title_text="False Positive Rate", range=[0, 1])
fig_roc.update_yaxes(title_text="True Positive Rate", range=[0, 1], col=1)
fig_roc.update_layout(height=400, width=1200, title="AUC-ROC Curves by Anomaly Type")
fig_roc.show()